In [3]:

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [4]:

import os
import sys
import torch
import numpy as np

# Paths
DATA_DIR = "/content/drive/MyDrive/T24_Dataset"
PROJECT_DIR = "/content/drive/MyDrive/T24_VAE_Results"

# New folder for the six experiments
EXPERIMENT_DIR = os.path.join(
    PROJECT_DIR, "VAE_6_Experiments"
)

os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# Import the VAE we already built
sys.path.insert(0, PROJECT_DIR)

from vae import VAE, vae_loss

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Shared settings
EPOCHS = 20
BATCH_SIZE = 128
LEARNING_RATE = 0.001
SEED = 42
WARMUP_EPOCHS = 10

# Six experiments
experiments = [
    {
        "name": "E1_Baseline",
        "latent_dim": 64,
        "beta": 1.0,
        "warmup": False
    },
    {
        "name": "E2_Latent32",
        "latent_dim": 32,
        "beta": 1.0,
        "warmup": False
    },
    {
        "name": "E3_Latent128",
        "latent_dim": 128,
        "beta": 1.0,
        "warmup": False
    },
    {
        "name": "E4_Beta0_5",
        "latent_dim": 64,
        "beta": 0.5,
        "warmup": False
    },
    {
        "name": "E5_Beta2",
        "latent_dim": 64,
        "beta": 2.0,
        "warmup": False
    },
    {
        "name": "E6_Warmup",
        "latent_dim": 64,
        "beta": 1.0,
        "warmup": True
    }
]

# Check the model file
test_model = VAE(latent_dim=64).to(device)

print("GPU:", device)
print("Number of experiments:", len(experiments))

for exp in experiments:
    print(
        exp["name"],
        "| Latent:", exp["latent_dim"],
        "| Beta:", exp["beta"],
        "| Warmup:", exp["warmup"]
    )

print("Experiments setup ready!")


GPU: cuda
Number of experiments: 6
E1_Baseline | Latent: 64 | Beta: 1.0 | Warmup: False
E2_Latent32 | Latent: 32 | Beta: 1.0 | Warmup: False
E3_Latent128 | Latent: 128 | Beta: 1.0 | Warmup: False
E4_Beta0_5 | Latent: 64 | Beta: 0.5 | Warmup: False
E5_Beta2 | Latent: 64 | Beta: 2.0 | Warmup: False
E6_Warmup | Latent: 64 | Beta: 1.0 | Warmup: True
Experiments setup ready!


In [5]:

import os
import json
import time
import random
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

# ===================================
# 1. LOAD DATASET
# ===================================

images = np.load(DATA_DIR + "/images.npy")
train_indices = np.load(DATA_DIR + "/train_indices.npy")
val_indices = np.load(DATA_DIR + "/val_indices.npy")

class VAEDataset(Dataset):
    def __init__(self, images, indices):
        self.images = images
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image = self.images[self.indices[idx]]
        image = image.astype(np.float32) / 255.0

        return torch.from_numpy(
            image.transpose(2, 0, 1).copy()
        )

train_data = VAEDataset(images, train_indices)
val_data = VAEDataset(images, val_indices)

# ===================================
# 2. TRAIN ONE EXPERIMENT
# ===================================

def run_experiment(config):

    name = config["name"]
    latent_dim = config["latent_dim"]
    beta = config["beta"]
    warmup = config["warmup"]

    run_dir = os.path.join(EXPERIMENT_DIR, name)
    os.makedirs(run_dir, exist_ok=True)

    ckpt_path = os.path.join(run_dir, "checkpoint.pth")
    history_path = os.path.join(run_dir, "history.json")

    # Same initial random seed for every run
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if device.type == "cuda":
        torch.cuda.manual_seed_all(SEED)

    # Independent, reproducible data shuffling
    loader_generator = torch.Generator()
    loader_generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=loader_generator,
        num_workers=0
    )

    val_loader = DataLoader(
        val_data,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    # Fresh model for each experiment
    model = VAE(latent_dim=latent_dim).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    start_epoch = 0
    global_step = 0
    history = []
    total_time = 0.0
    peak_memory = 0.0

    # Resume if experiment was interrupted
    if os.path.exists(ckpt_path):
        ckpt = torch.load(
            ckpt_path,
            map_location="cpu",
            weights_only=True
        )

        assert ckpt["config"] == config

        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])

        start_epoch = ckpt["epoch"]
        global_step = ckpt["global_step"]
        history = ckpt["history"]
        total_time = ckpt["training_time"]
        peak_memory = ckpt["peak_memory_mb"]

        torch.set_rng_state(ckpt["torch_rng_state"])
        loader_generator.set_state(
            ckpt["loader_rng_state"]
        )

        if device.type == "cuda":
            torch.cuda.set_rng_state_all(
                ckpt["cuda_rng_states"]
            )

        print("Resuming", name, "from epoch", start_epoch)

    print("\nSTARTING:", name)

    # ===================================
    # 3. EPOCH LOOP
    # ===================================

    for epoch in range(start_epoch, EPOCHS):

        if device.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        start_time = time.perf_counter()

        # KL warm-up schedule
        if warmup:
            current_beta = beta * min(
                1.0, (epoch + 1) / WARMUP_EPOCHS
            )
        else:
            current_beta = beta

        model.train()

        train_total = 0.0
        train_recon = 0.0
        train_kl = 0.0

        for batch in train_loader:
            batch = batch.to(device)

            optimizer.zero_grad(set_to_none=True)

            output, mean, logvar = model(batch)

            loss, recon, kl = vae_loss(
                output,
                batch,
                mean,
                logvar,
                beta=current_beta
            )

            if not torch.isfinite(loss).item():
                raise ValueError("Non-finite loss")

            loss.backward()
            optimizer.step()

            n = batch.size(0)
            train_total += loss.item() * n
            train_recon += recon.item() * n
            train_kl += kl.item() * n
            global_step += 1

        train_total /= len(train_data)
        train_recon /= len(train_data)
        train_kl /= len(train_data)

        # ===================================
        # 4. VALIDATION
        # ===================================

        model.eval()

        val_recon_sum = 0.0
        val_kl_sum = 0.0
        val_squared_error = 0.0
        all_means = []

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)

                output, mean, logvar = model(batch)

                _, recon, kl = vae_loss(
                    output, batch, mean, logvar, beta=1.0
                )

                n = batch.size(0)

                val_recon_sum += recon.item() * n
                val_kl_sum += kl.item() * n

                # Stable reconstruction for pixel MSE
                deterministic_output = model.decoder(mean)

                val_squared_error += (
                    (deterministic_output - batch).pow(2)
                ).sum().item()

                all_means.append(mean.cpu())

        val_recon = val_recon_sum / len(val_data)
        val_kl = val_kl_sum / len(val_data)

        # Always evaluate negative ELBO with beta=1
        negative_elbo = val_recon + val_kl

        val_mse = val_squared_error / (
            len(val_data) * 3 * 64 * 64
        )

        # Active latent units
        means = torch.cat(all_means, dim=0)
        latent_variance = means.var(
            dim=0, unbiased=False
        )

        active_units = int(
            (latent_variance > 0.01).sum().item()
        )

        if device.type == "cuda":
            torch.cuda.synchronize()

        epoch_time = time.perf_counter() - start_time
        total_time += epoch_time

        memory_mb = (
            torch.cuda.max_memory_allocated() / (1024**2)
            if device.type == "cuda" else 0.0
        )

        peak_memory = max(peak_memory, memory_mb)

        # ===================================
        # 5. SAVE RESULTS
        # ===================================

        record = {
            "epoch": epoch + 1,
            "beta_used": current_beta,
            "train_loss": train_total,
            "train_recon": train_recon,
            "train_kl": train_kl,
            "val_recon": val_recon,
            "val_kl": val_kl,
            "val_mse": val_mse,
            "negative_elbo_beta1": negative_elbo,
            "active_units": active_units,
            "epoch_time": epoch_time,
            "peak_memory_mb": memory_mb
        }

        history.append(record)

        torch.save({
            "epoch": epoch + 1,
            "global_step": global_step,
            "config": config,
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "history": history,
            "training_time": total_time,
            "peak_memory_mb": peak_memory,
            "torch_rng_state": torch.get_rng_state(),
            "loader_rng_state": loader_generator.get_state(),
            "cuda_rng_states": (
                torch.cuda.get_rng_state_all()
                if device.type == "cuda" else []
            )
        }, ckpt_path)

        with open(history_path, "w") as f:
            json.dump(history, f, indent=4)

        print(
            f"{name} | Epoch {epoch+1}/{EPOCHS} | "
            f"VAL MSE: {val_mse:.5f} | "
            f"KL: {val_kl:.2f} | "
            f"ELBO: {negative_elbo:.2f} | "
            f"Active: {active_units}/{latent_dim}"
        )

    print("COMPLETED:", name)
    return history


# ===================================
# 6. RUN FIRST EXPERIMENT
# ===================================

# = run_experiment(experiments[0])


In [6]:

# Experiment 2: Latent Dimension = 32

history_e2 = run_experiment(experiments[1])



STARTING: E2_Latent32
E2_Latent32 | Epoch 1/20 | VAL MSE: 0.04049 | KL: 26.76 | ELBO: 536.39 | Active: 32/32
E2_Latent32 | Epoch 2/20 | VAL MSE: 0.03162 | KL: 33.04 | ELBO: 432.60 | Active: 32/32
E2_Latent32 | Epoch 3/20 | VAL MSE: 0.02729 | KL: 36.76 | ELBO: 382.49 | Active: 32/32
E2_Latent32 | Epoch 4/20 | VAL MSE: 0.02532 | KL: 35.22 | ELBO: 357.44 | Active: 32/32
E2_Latent32 | Epoch 5/20 | VAL MSE: 0.02344 | KL: 38.44 | ELBO: 338.91 | Active: 32/32
E2_Latent32 | Epoch 6/20 | VAL MSE: 0.02293 | KL: 39.96 | ELBO: 334.48 | Active: 32/32
E2_Latent32 | Epoch 7/20 | VAL MSE: 0.02212 | KL: 41.38 | ELBO: 326.49 | Active: 32/32
E2_Latent32 | Epoch 8/20 | VAL MSE: 0.02166 | KL: 43.43 | ELBO: 322.77 | Active: 32/32
E2_Latent32 | Epoch 9/20 | VAL MSE: 0.02129 | KL: 45.18 | ELBO: 319.67 | Active: 32/32
E2_Latent32 | Epoch 10/20 | VAL MSE: 0.02139 | KL: 43.94 | ELBO: 321.13 | Active: 32/32
E2_Latent32 | Epoch 11/20 | VAL MSE: 0.02125 | KL: 45.60 | ELBO: 319.11 | Active: 32/32
E2_Latent32 | Epoc

In [7]:

# Experiment 3: Latent Dimension = 128

history_e3 = run_experiment(experiments[2])



STARTING: E3_Latent128
E3_Latent128 | Epoch 1/20 | VAL MSE: 0.04739 | KL: 21.04 | ELBO: 615.40 | Active: 128/128
E3_Latent128 | Epoch 2/20 | VAL MSE: 0.03277 | KL: 40.98 | ELBO: 463.19 | Active: 128/128
E3_Latent128 | Epoch 3/20 | VAL MSE: 0.03052 | KL: 41.36 | ELBO: 416.06 | Active: 128/128
E3_Latent128 | Epoch 4/20 | VAL MSE: 0.02670 | KL: 38.84 | ELBO: 402.98 | Active: 128/128
E3_Latent128 | Epoch 5/20 | VAL MSE: 0.02478 | KL: 46.07 | ELBO: 360.41 | Active: 128/128
E3_Latent128 | Epoch 6/20 | VAL MSE: 0.02376 | KL: 46.42 | ELBO: 346.76 | Active: 128/128
E3_Latent128 | Epoch 7/20 | VAL MSE: 0.02216 | KL: 48.00 | ELBO: 335.02 | Active: 128/128
E3_Latent128 | Epoch 8/20 | VAL MSE: 0.02141 | KL: 46.13 | ELBO: 326.36 | Active: 128/128
E3_Latent128 | Epoch 9/20 | VAL MSE: 0.02134 | KL: 50.75 | ELBO: 321.57 | Active: 127/128
E3_Latent128 | Epoch 10/20 | VAL MSE: 0.02063 | KL: 48.97 | ELBO: 316.82 | Active: 125/128
E3_Latent128 | Epoch 11/20 | VAL MSE: 0.01965 | KL: 51.91 | ELBO: 312.47 | 

In [8]:

# Experiment 4: Beta = 0.5

history_e4 = run_experiment(experiments[3])



STARTING: E4_Beta0_5
E4_Beta0_5 | Epoch 1/20 | VAL MSE: 0.04576 | KL: 34.46 | ELBO: 604.01 | Active: 64/64
E4_Beta0_5 | Epoch 2/20 | VAL MSE: 0.03084 | KL: 57.36 | ELBO: 446.64 | Active: 64/64
E4_Beta0_5 | Epoch 3/20 | VAL MSE: 0.02775 | KL: 58.81 | ELBO: 411.44 | Active: 64/64
E4_Beta0_5 | Epoch 4/20 | VAL MSE: 0.02499 | KL: 58.90 | ELBO: 374.89 | Active: 64/64
E4_Beta0_5 | Epoch 5/20 | VAL MSE: 0.02307 | KL: 62.88 | ELBO: 361.90 | Active: 64/64
E4_Beta0_5 | Epoch 6/20 | VAL MSE: 0.02301 | KL: 65.90 | ELBO: 354.38 | Active: 64/64
E4_Beta0_5 | Epoch 7/20 | VAL MSE: 0.02094 | KL: 67.02 | ELBO: 335.83 | Active: 64/64
E4_Beta0_5 | Epoch 8/20 | VAL MSE: 0.02065 | KL: 65.77 | ELBO: 327.20 | Active: 64/64
E4_Beta0_5 | Epoch 9/20 | VAL MSE: 0.01970 | KL: 66.98 | ELBO: 319.35 | Active: 64/64
E4_Beta0_5 | Epoch 10/20 | VAL MSE: 0.01929 | KL: 70.43 | ELBO: 317.52 | Active: 64/64
E4_Beta0_5 | Epoch 11/20 | VAL MSE: 0.01904 | KL: 69.96 | ELBO: 315.04 | Active: 64/64
E4_Beta0_5 | Epoch 12/20 | VAL

In [9]:

# Experiment 5: Beta = 2

history_e5 = run_experiment(experiments[4])



STARTING: E5_Beta2
E5_Beta2 | Epoch 1/20 | VAL MSE: 0.04608 | KL: 16.93 | ELBO: 605.77 | Active: 64/64
E5_Beta2 | Epoch 2/20 | VAL MSE: 0.03265 | KL: 24.91 | ELBO: 447.24 | Active: 64/64
E5_Beta2 | Epoch 3/20 | VAL MSE: 0.03148 | KL: 25.29 | ELBO: 411.98 | Active: 64/64
E5_Beta2 | Epoch 4/20 | VAL MSE: 0.03067 | KL: 23.66 | ELBO: 400.70 | Active: 64/64
E5_Beta2 | Epoch 5/20 | VAL MSE: 0.02555 | KL: 28.24 | ELBO: 362.74 | Active: 64/64
E5_Beta2 | Epoch 6/20 | VAL MSE: 0.02400 | KL: 30.27 | ELBO: 347.69 | Active: 64/64
E5_Beta2 | Epoch 7/20 | VAL MSE: 0.02345 | KL: 29.84 | ELBO: 343.11 | Active: 64/64
E5_Beta2 | Epoch 8/20 | VAL MSE: 0.02329 | KL: 31.98 | ELBO: 338.83 | Active: 62/64
E5_Beta2 | Epoch 9/20 | VAL MSE: 0.02351 | KL: 31.56 | ELBO: 338.92 | Active: 63/64
E5_Beta2 | Epoch 10/20 | VAL MSE: 0.02178 | KL: 34.66 | ELBO: 326.03 | Active: 64/64
E5_Beta2 | Epoch 11/20 | VAL MSE: 0.02173 | KL: 31.51 | ELBO: 327.79 | Active: 64/64
E5_Beta2 | Epoch 12/20 | VAL MSE: 0.02151 | KL: 34.03 

In [10]:

# Experiment 6: KL Warm-up ON

history_e6 = run_experiment(experiments[5])



STARTING: E6_Warmup
E6_Warmup | Epoch 1/20 | VAL MSE: 0.04462 | KL: 78.33 | ELBO: 630.36 | Active: 64/64
E6_Warmup | Epoch 2/20 | VAL MSE: 0.02977 | KL: 92.54 | ELBO: 464.53 | Active: 64/64
E6_Warmup | Epoch 3/20 | VAL MSE: 0.02726 | KL: 79.83 | ELBO: 422.02 | Active: 64/64
E6_Warmup | Epoch 4/20 | VAL MSE: 0.02474 | KL: 71.87 | ELBO: 383.18 | Active: 64/64
E6_Warmup | Epoch 5/20 | VAL MSE: 0.02350 | KL: 68.91 | ELBO: 372.46 | Active: 64/64
E6_Warmup | Epoch 6/20 | VAL MSE: 0.02242 | KL: 62.97 | ELBO: 347.97 | Active: 64/64
E6_Warmup | Epoch 7/20 | VAL MSE: 0.02094 | KL: 63.73 | ELBO: 333.40 | Active: 64/64
E6_Warmup | Epoch 8/20 | VAL MSE: 0.02065 | KL: 59.70 | ELBO: 327.22 | Active: 64/64
E6_Warmup | Epoch 9/20 | VAL MSE: 0.02105 | KL: 59.84 | ELBO: 331.32 | Active: 64/64
E6_Warmup | Epoch 10/20 | VAL MSE: 0.02028 | KL: 57.97 | ELBO: 328.69 | Active: 64/64
E6_Warmup | Epoch 11/20 | VAL MSE: 0.01941 | KL: 55.07 | ELBO: 313.54 | Active: 64/64
E6_Warmup | Epoch 12/20 | VAL MSE: 0.02022